# Portfolio Optimizer — Demo

Demonstrates the full pipeline: data loading, optimization (GMV / Max-Sharpe / efficient frontier), risk evaluation, stress testing, forward stepwise asset selection, and comparing portfolios.

See `../README.md` for methodology notes, metric interpretation guidance, and known limitations.

In [1]:
import sys
sys.path.append("..")

from portfolio_optimizer import (
    run_portfolio_analysis,
    forward_stepwise_selection,
    compare_all_portfolios,
)

## 1. Build a manually chosen portfolio

Loads data, solves for the GMV and Max-Sharpe portfolios, plots the efficient frontier, and runs the full risk evaluation + stress test. `save_as` registers this portfolio for the comparison in section 4.

In [2]:
results = run_portfolio_analysis(["VXUS", "BND", "GLD", "VNQ"], save_as="auto")

[*********************100%***********************]  3 of 4 completed


Asset-level statistics (annualized):
                   BND     GLD     VNQ    VXUS
Expected Return  2.31%   8.57%   9.33%   8.16%
Volatility       4.86%  16.77%  19.81%  17.72%

Portfolio weights:
                      BND     GLD     VNQ   VXUS
GMV Weight         94.37%   0.00%   0.00%  5.63%
Max-Sharpe Weight   0.00%  52.03%  41.85%  6.12%

Portfolio summary:
           Expected Return Volatility Sharpe
GMV                  2.64%      4.76%  -0.43
Max-Sharpe           8.87%     13.32%   0.31

RISK EVALUATION — Max-Sharpe portfolio

Expected return: 8.87% | Volatility: 13.32% | Sharpe: 0.31

--- Tail risk (95% confidence, $1,000,000 portfolio) ---
Parametric VaR:   $130,466  (13.05%)
Parametric CVaR:  $186,131  (18.61%)
Historical VaR:   $203,867  (20.39%)
Historical CVaR:  $318,587  (31.86%)

--- Path risk ---
Max drawdown: -24.02%

--- Risk-adjusted return ratios ---
Sortino ratio: 0.40   Calmar ratio: 0.37

--- Distribution shape ---
Skewness: -0.58   Excess kurtosis: 9.07

--- Di

## 2. Select assets from a larger candidate universe

Greedy forward stepwise selection: starts with the single best asset, then repeatedly adds whichever remaining candidate improves the Max-Sharpe ratio the most, until no candidate improves it by at least `min_improvement` (or `max_assets` is reached).

`min_improvement` matters here: without a real threshold, the algorithm would keep adding assets even for negligible or purely numerical-noise gains, since adding an asset can never mathematically *hurt* the achievable Sharpe ratio in a long-only optimizer — only a minimum-improvement cutoff actually stops it at a meaningful point.

In [3]:
candidates = ["VXUS", "BND", "GLD", "VNQ", "TLT", "DBC", "VWO", "SHY"]
selected_tickers, selection_history = forward_stepwise_selection(candidates)

[*********************100%***********************]  8 of 8 completed


FORWARD STEPWISE SELECTION
(minimum Sharpe improvement to keep adding: 0.005)
Step 1: added VNQ      -> Sharpe = 0.234 (+inf)
Step 2: added GLD      -> Sharpe = 0.313 (+0.0784)

Best remaining candidate (VXUS) only improves Sharpe by 0.0006, below the 0.005 threshold — stopping.

Final selected assets: ['VNQ', 'GLD']
Final Sharpe ratio: 0.313
(See README.md "Interpreting the risk metrics" for guidance on reading the per-step improvement.)


## 3. Evaluate the selected portfolio

Feed the output of forward selection straight back into the pipeline — again saved for the comparison below.

In [4]:
selected_results = run_portfolio_analysis(selected_tickers, save_as="auto")

[*********************100%***********************]  2 of 2 completed

Asset-level statistics (annualized):
                    GLD     VNQ
Expected Return   9.14%  10.60%
Volatility       16.76%  20.30%

Portfolio weights:
                      GLD     VNQ
GMV Weight         60.64%  39.36%
Max-Sharpe Weight  51.90%  48.10%

Portfolio summary:
           Expected Return Volatility Sharpe
GMV                  9.71%     13.61%   0.37
Max-Sharpe           9.84%     13.78%   0.37

RISK EVALUATION — Max-Sharpe portfolio

Expected return: 9.84% | Volatility: 13.78% | Sharpe: 0.37

--- Tail risk (95% confidence, $1,000,000 portfolio) ---
Parametric VaR:   $128,317  (12.83%)
Parametric CVaR:  $185,911  (18.59%)
Historical VaR:   $214,397  (21.44%)
Historical CVaR:  $328,701  (32.87%)

--- Path risk ---
Max drawdown: -24.64%

--- Risk-adjusted return ratios ---
Sortino ratio: 0.48   Calmar ratio: 0.40

--- Distribution shape ---
Skewness: -0.55   Excess kurtosis: 8.24

--- Diversification ---
Diversification ratio: 1.34

(See README.md "Interpreting the risk metri

## 4. Compare all saved portfolios

Pulls together every portfolio saved via `save_as` above into one side-by-side table. Add more `run_portfolio_analysis(..., save_as="...")` calls above (e.g. a different manual mix) to include them here too — no separate comparison call needed per pair.

In [5]:
compare_all_portfolios()

Comparison (Max-Sharpe portfolios):
                      BND+GLD+VNQ+VXUS   GLD+VNQ
Expected Return                  8.87%     9.84%
Volatility                      13.32%    13.78%
Sharpe                            0.31      0.37
Parametric VaR                $130,466  $128,317
Parametric CVaR               $186,131  $185,911
Historical VaR                $203,867  $214,397
Historical CVaR               $318,587  $328,701
Max Drawdown                   -24.02%   -24.64%
Sortino Ratio                     0.40      0.48
Calmar Ratio                      0.37      0.40
Skewness                         -0.58     -0.55
Excess Kurtosis                   9.07      8.24
Diversification Ratio             1.36      1.34
# Assets                             4         2

Highest Sharpe: GLD+VNQ (0.374)
